In [2]:
# File: q3_risk_profiling_kimberly.R
# Purpose: Identify volcanoes that pose the greatest hazard based on eruption
# frequency and nearby population exposure
# Author: Jan McConnell/Kimberly Thomas
# Date: 2025-07-27
# Course: DS520 – Data Mining
# Question: Q3 – Volcanic Risk Profiling

###############################################################################

In [3]:
# Install packages
install.packages("readxl")
install.packages("dplyr")
install.packages("ggplot2")
install.packages("psych")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mnormt’, ‘GPArotation’




In [5]:
# Load necessary libraries
library(readxl)     # for reading Excel files
library(dplyr)      # for data manipulation
library(ggplot2)    # for visualization
library(psych)      # for classification


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘psych’


The following objects are masked from ‘package:ggplot2’:

    %+%, alpha




In [6]:
# Examine sheet names (confirm expected structure)
excel_sheets("GVP_Eruption_Search_Result.xlsx")
excel_sheets("GVP_Volcano_List_Holocene_202507152349.xlsx")

[1] "Eruption List"

[1] "Holocene Volcano List"

In [7]:
# Load data with specified column types to avoid warnings
eruptions <- read_excel("GVP_Eruption_Search_Result.xlsx", sheet = "Eruption List", col_types = "text")
volcanoes <- read_excel("GVP_Volcano_List_Holocene_202507152349.xlsx", sheet = "Holocene Volcano List", col_types = "text")

In [8]:
# Preview file structure
cat("\n--- Eruption File Columns ---\n")
print(colnames(eruptions))
cat("\nEruption File Sample:\n")
print(head(eruptions, 3))

cat("\n--- Volcano File Columns ---\n")
print(colnames(volcanoes))
cat("\nVolcano File Sample:\n")
print(head(volcanoes, 3))


--- Eruption File Columns ---
 [1] "Volcano_Number"           "Volcano_Name"            
 [3] "Eruption_Number"          "Eruption_Category"       
 [5] "Area_of_Activity"         "VEI"                     
 [7] "VEI_Modifier"             "Start_Year_Modifier"     
 [9] "Start_Year"               "Start_Year_Uncertainty"  
[11] "Start_Month"              "Start_Day_Modifier"      
[13] "Start_Day"                "Start_Day_Uncertainty"   
[15] "Evidence_Method_(dating)" "End_Year_Modifier"       
[17] "End_Year"                 "End_Year_Uncertainty"    
[19] "End_Month"                "End_Day_Modifier"        
[21] "End_Day"                  "End_Day_Uncertainty"     
[23] "Latitude"                 "Longitude"               

Eruption File Sample:
# A tibble: 3 × 24
  Volcano_Number Volcano_Name Eruption_Number Eruption_Category Area_of_Activity
  <chr>          <chr>        <chr>           <chr>             <chr>           
1 300130         Karymsky     22609           Confirmed E

In [9]:
# Count number of confirmed eruptions per volcano
eruption_counts <- eruptions %>%
  group_by(Volcano_Number) %>%
  summarise(eruption_count = n(), .groups = "drop")

In [10]:
# Select key volcano attributes
volcano_info <- volcanoes %>%
  select(
    Volcano_Number,
    Volcano_Name,
    Country,
    `Elevation_(m)`,
    Tectonic_Setting
  )

In [12]:
# Join eruption counts with volcano info
volcano_risk <- volcano_info %>%
  left_join(eruption_counts, by = "Volcano_Number") %>%
  mutate(eruption_count = ifelse(is.na(eruption_count), 0, eruption_count))

In [13]:
# Compute simple risk score based on eruption frequency
volcano_risk <- volcano_risk %>%
  mutate(risk_score = eruption_count)

In [14]:
# Identify top 10 volcanoes by eruption frequency
top_risks <- volcano_risk %>%
  arrange(desc(risk_score)) %>%
  head(10)

In [15]:
# Print top results to console
cat("\nTop 10 Volcanoes by Eruption Frequency:\n")
print(top_risks)


Top 10 Volcanoes by Eruption Frequency:
# A tibble: 10 × 7
   Volcano_Number Volcano_Name          Country `Elevation_(m)` Tectonic_Setting
   <chr>          <chr>                 <chr>   <chr>           <chr>           
 1 233020         Fournaise, Piton de … France  2632            Intraplate / Oc…
 2 282110         Asosan                Japan   1592            Subduction zone…
 3 357120         Villarrica            Chile   2847            Subduction zone…
 4 211060         Etna                  Italy   3357            Subduction zone…
 5 283110         Asamayama             Japan   2568            Subduction zone…
 6 372030         Katla                 Iceland 1490            Rift zone / Oce…
 7 300260         Klyuchevskoy          Russia  4754            Subduction zone…
 8 332020         Mauna Loa             United… 4170            Intraplate / Oc…
 9 263250         Merapi                Indone… 2910            Subduction zone…
10 300270         Sheveluch             Russia  3

In [16]:
# Export full volcano risk data to CSV
write.csv(
  volcano_risk,
  "/volcano_risk_profile.csv",
  row.names = FALSE
)

In [17]:
# Join average VEI with the volcano risk data in the selected cell (Google Colab prompt)

# Ensure VEI is numeric and calculate average VEI per volcano
eruptions$VEI_numeric <- as.numeric(eruptions$VEI)
average_vei <- eruptions %>%
  group_by(Volcano_Number) %>%
  summarise(average_vei = mean(VEI_numeric, na.rm = TRUE), .groups = "drop")

# Join average VEI with the volcano risk data
volcano_risk_vei <- volcano_risk %>%
  left_join(average_vei, by = "Volcano_Number")

# Display the head of the resulting dataframe
cat("\nVolcano Risk Data with Average VEI:\n")
print(head(volcano_risk_vei))


Volcano Risk Data with Average VEI:
# A tibble: 6 × 8
  Volcano_Number Volcano_Name           Country `Elevation_(m)` Tectonic_Setting
  <chr>          <chr>                  <chr>   <chr>           <chr>           
1 210010         West Eifel Volcanic F… Germany 600             Rift zone / Con…
2 210020         Chaine des Puys        France  1464            Rift zone / Con…
3 210030         Olot Volcanic Field    Spain   893             Intraplate / Co…
4 210040         Calatrava Volcanic Fi… Spain   1117            Intraplate / Co…
5 211004         Colli Albani           Italy   949             Subduction zone…
6 211010         Campi Flegrei          Italy   458             Subduction zone…
# ℹ 3 more variables: eruption_count <dbl>, risk_score <dbl>, average_vei <dbl>


In [18]:
# Identify top 10 volcanoes by average VEI
top_vei <- volcano_risk_vei %>%
  arrange(desc(average_vei)) %>%
  head(10)

# Print top results to console
cat("\nTop 10 Volcanoes by Average VEI:\n")
print(top_vei)


Top 10 Volcanoes by Average VEI:
# A tibble: 10 × 8
   Volcano_Number Volcano_Name          Country `Elevation_(m)` Tectonic_Setting
   <chr>          <chr>                 <chr>   <chr>           <chr>           
 1 300023         Kurile Lake           Russia  81              Subduction zone…
 2 355210         Blanco, Cerro         Argent… 4670            Subduction zone…
 3 222060         Menengai              Kenya   2278            Rift zone / Con…
 4 242021         Macauley              New Ze… 238             Subduction zone…
 5 290041         Moekeshiwan [Lvinaya… Japan … 495             Subduction zone…
 6 306030         Ulleungdo             South … 984             Intraplate / Co…
 7 312080         Black Peak            United… 1032            Subduction zone…
 8 312180         Novarupta             United… 841             Subduction zone…
 9 315030         Churchill             United… 5005            Intraplate / Co…
10 352060         Quilotoa              Ecuador 3914    

In [19]:
# Save the volcano_risk_vei dataframe to a CSV file
write.csv(
  volcano_risk_vei,
  "volcano_risk_profile_with_vei.csv",
  row.names = FALSE
)

In [23]:
# Create and save a bar plot of average VEI for the top 10 volcanoes (Google Colab prompt)
pdf("Top 10 Volcanoes by Average VEI.pdf")

ggplot(top_vei, aes(x = reorder(Volcano_Name, average_vei), y = average_vei)) +
  geom_bar(stat = "identity", fill = "skyblue") +
  labs(title = "Top 10 Volcanoes by Average VEI",
       x = "Volcano Name",
       y = "Average VEI") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) # Rotate x-axis labels for readability

dev.off()

agg_record_2083740867 
                    2